# Lab

Plan a purchasing batch: compare calculation methods, order reusable packs, then reconcile one-use invoices. Each task builds on the same planning workflow. Use Edit and Run. Copy the relevant complete implementation from the lessons when writing your answer; each solution includes its own setup. Sign in before saving answers.

## 1. Measure Repeated Work

```{index} 1. Measure Repeated Work
```

At n=8, compare naive and memoized Fibonacci with the same input and counting policy. Report value, calls, and non-base computations. Explain why cached calls still count.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int n=8;
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int n=8;
var raw=FibPlanning.Fibonacci(n,false);var memo=FibPlanning.Fibonacci(n);
Console.WriteLine($"value={raw.Value}, naive calls={raw.Calls}, memo calls={memo.Calls}, computations={memo.Computed}");
if(raw.Value!=memo.Value||memo.Computed!=n-1)throw new Exception("Memo contract failed.");
public static class FibPlanning
{
    public record FibResult(long Value,int Calls,int Hits,int Computed);
    // F(0)=0, F(1)=1. F(92) is the last value representable by long.
    public static FibResult Fibonacci(int n,bool memoized=true)
    {
        if(n<0||n>92||(!memoized&&n>25))throw new ArgumentOutOfRangeException(nameof(n));
        var cache=new Dictionary<int,long>();int calls=0,hits=0,computed=0;
        long Solve(int k)
        {
            calls++;
            if(k<=1)return k;
            if(memoized&&cache.TryGetValue(k,out long saved)){hits++;return saved;}
            long value=checked(Solve(k-1)+Solve(k-2));computed++;
            if(memoized)cache[k]=value;
            return value;
        }
        return new FibResult(Solve(n),calls,hits,computed);
    }
    public static long[] FibonacciTable(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        var table=new long[n+1];if(n>=1)table[1]=1;
        for(int i=2;i<=n;i++)table[i]=checked(table[i-1]+table[i-2]);
        return table;
    }
    public static long FibonacciRolling(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        if(n==0)return 0;
        long before=0,current=1;
        for(int i=2;i<=n;i++)(before,current)=(current,checked(before+current));
        return current;
    }
}


value=21, naive calls=67, memo calls=15, computations=7


## 2. Choose Retained State

```{index} 2. Choose Retained State
```

Compute n=8 with table and rolling state. Check agreement with the memoized value. Test zero. Explain which method can return all intermediate entries and which keeps constant auxiliary storage.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int n=8;
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int n=8;
var table=FibPlanning.FibonacciTable(n);var rolling=FibPlanning.FibonacciRolling(n);
if(table[n]!=rolling||rolling!=FibPlanning.Fibonacci(n).Value)throw new Exception("DP disagreement.");
Console.WriteLine($"table=[{string.Join(",",table)}], rolling={rolling}, zero={FibPlanning.FibonacciTable(0)[0]}");
public static class FibPlanning
{
    public record FibResult(long Value,int Calls,int Hits,int Computed);
    // F(0)=0, F(1)=1. F(92) is the last value representable by long.
    public static FibResult Fibonacci(int n,bool memoized=true)
    {
        if(n<0||n>92||(!memoized&&n>25))throw new ArgumentOutOfRangeException(nameof(n));
        var cache=new Dictionary<int,long>();int calls=0,hits=0,computed=0;
        long Solve(int k)
        {
            calls++;
            if(k<=1)return k;
            if(memoized&&cache.TryGetValue(k,out long saved)){hits++;return saved;}
            long value=checked(Solve(k-1)+Solve(k-2));computed++;
            if(memoized)cache[k]=value;
            return value;
        }
        return new FibResult(Solve(n),calls,hits,computed);
    }
    public static long[] FibonacciTable(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        var table=new long[n+1];if(n>=1)table[1]=1;
        for(int i=2;i<=n;i++)table[i]=checked(table[i-1]+table[i-2]);
        return table;
    }
    public static long FibonacciRolling(int n)
    {
        if(n<0||n>92)throw new ArgumentOutOfRangeException(nameof(n));
        if(n==0)return 0;
        long before=0,current=1;
        for(int i=2;i<=n;i++)(before,current)=(current,checked(before+current));
        return current;
    }
}


table=[0,1,1,2,3,5,8,13,21], rolling=21, zero=0


## 3. Order the Fewest Packs

```{index} 3. Order the Fewest Packs
```

Pack sizes 1,3,4 are reusable. Request six units. Recover and validate a minimum-count witness, compare largest-first’s three packs, and test an unreachable target seven with sizes 4,6.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] sizes={1,3,4};int target=6;
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] sizes={1,3,4};int target=6;
var p=CoinPlanning.MinimumCoins(sizes,target);
if(p.Count!=2||p.Coins.Length!=p.Count||p.Coins.Sum()!=target||p.Coins.Any(x=>!sizes.Contains(x)))throw new Exception("Pack witness failed.");
Console.WriteLine($"minimum={p.Count}, witness={string.Join("+",p.Coins)}; largest-first count=3");
Console.WriteLine($"seven with 4/6 reachable={CoinPlanning.MinimumCoins(new[]{4,6},7).Count is not null}");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}


minimum=2, witness=3+3; largest-first count=3
seven with 4/6 reachable=False


## 4. Reconcile One-Use Invoices

```{index} 4. Reconcile One-Use Invoices
```

Invoices 8,−3,4,4 include a credit and two distinct equal-valued records. Find positions totaling five, validate their sum and uniqueness, and contrast reusable four-packs with only two four-unit invoice records for target twelve.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] invoices={8,-3,4,4};long target=5;
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] invoices={8,-3,4,4};long target=5;
var r=SubsetPlanning.FindSubset(invoices,target);
if(r.Indices is null||r.Indices.Distinct().Count()!=r.Indices.Length||r.Indices.Sum(i=>(long)invoices[i])!=target)throw new Exception("Invoice witness failed.");
Console.WriteLine($"IDs={string.Join(",",r.Indices)}, total={r.Indices.Sum(i=>(long)invoices[i])}");
Console.WriteLine($"four-packs count={CoinPlanning.MinimumCoins(new[]{4},12).Count}; two invoices feasible={SubsetPlanning.FindSubset(new[]{4,4},12).Indices is not null}");
public static class CoinPlanning
{
    public record CoinPlan(int? Count,int[] Coins,int?[] Best);
    // Unlimited positive denominations. A target cap bounds table allocation.
    public static CoinPlan MinimumCoins(int[] denominations,int target)
    {
        ArgumentNullException.ThrowIfNull(denominations);
        if(target<0||target>1_000_000)throw new ArgumentOutOfRangeException(nameof(target));
        if(denominations.Any(c=>c<=0))throw new ArgumentException("Coins must be positive.");
        int[] coins=denominations.Distinct().OrderBy(c=>c).ToArray();
        var best=new int?[target+1];var last=new int[target+1];best[0]=0;
        for(int amount=1;amount<=target;amount++)
        {
            foreach(int coin in coins)
            {
                if(coin>amount)break;
                if(best[amount-coin] is not int count)continue;
                int candidate=count+1;
                if(best[amount] is null||candidate<best[amount])
                {best[amount]=candidate;last[amount]=coin;}
            }
        }
        var chosen=new List<int>();
        if(best[target] is not null)
            for(int remaining=target;remaining>0;remaining-=last[remaining])chosen.Add(last[remaining]);
        return new CoinPlan(best[target],chosen.ToArray(),best);
    }
}

public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}


IDs=0,1, total=5
four-packs count=3; two invoices feasible=False


## 5. Explain a Search Improvement

```{index} 5. Explain a Search Improvement
```

For twelve invoices each worth two and target eleven, compare uncached search, signed-bound pruning, and failed-state memoization on the same input. Verify infeasibility with exhaustive positions. Report calls/hits and explain why the state includes position.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

int[] invoices=Enumerable.Repeat(2,12).ToArray();long target=11;
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.

In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
int[] invoices=Enumerable.Repeat(2,12).ToArray();long target=11;
var raw=SubsetPlanning.FindSubset(invoices,target,false,false);
var pruned=SubsetPlanning.FindSubset(invoices,target,true,false);
var memo=SubsetPlanning.FindSubset(invoices,target,true,true);
bool reference=ChoicePlanning.Subsets(invoices.Length).Any(ids=>ids.Sum(i=>(long)invoices[i])==target);
if(reference||raw.Indices is not null||pruned.Indices is not null||memo.Indices is not null)throw new Exception("Infeasibility disagreement.");
Console.WriteLine($"raw calls={raw.Calls}, bounds calls={pruned.Calls}, memo calls={memo.Calls}, hits={memo.Hits}");
Console.WriteLine("one-use state=(position, remaining); exhaustive feasible=False");
public static class SubsetPlanning
{
    public record SubsetResult(int[]? Indices,int Calls,int Hits,int Prunes,int FailedStates);
    // Signed values are allowed; positions are distinct and each is used once.
    public static SubsetResult FindSubset(int[] values,long target,bool prune=true,bool memoize=true,int maxCalls=1_000_000)
    {
        ArgumentNullException.ThrowIfNull(values);
        if(values.Length>128)throw new ArgumentException("At most 128 positions in this teaching implementation.");
        if(maxCalls<1)throw new ArgumentOutOfRangeException(nameof(maxCalls));
        int[] data=(int[])values.Clone();int n=data.Length;
        var lower=new long[n+1];var upper=new long[n+1];
        for(int i=n-1;i>=0;i--)
        {lower[i]=lower[i+1]+Math.Min(0,data[i]);upper[i]=upper[i+1]+Math.Max(0,data[i]);}
        var failed=new HashSet<(int Index,long Remaining)>();var chosen=new List<int>();
        int calls=0,hits=0,prunes=0;int[]? answer=null;
        bool Search(int index,long remaining)
        {
            if(calls==maxCalls)throw new InvalidOperationException("Search budget exhausted; feasibility is unknown.");
            calls++;
            if(remaining==0){answer=chosen.ToArray();return true;}
            if(index==n)return false;
            if(prune&&(remaining<lower[index]||remaining>upper[index])){prunes++;return false;}
            var state=(index,remaining);
            if(memoize&&failed.Contains(state)){hits++;return false;}
            chosen.Add(index);bool found;
            try{found=Search(index+1,remaining-data[index]);}
            finally{chosen.RemoveAt(chosen.Count-1);}
            if(found||Search(index+1,remaining))return true;
            if(memoize)failed.Add(state);
            return false;
        }
        // Even an unpruned trace rejects targets outside the total signed range.
        if(target<lower[0]||target>upper[0]){calls=1;prunes=1;}
        else Search(0,target);
        if(chosen.Count!=0)throw new InvalidOperationException("Undo failed.");
        return new SubsetResult(answer,calls,hits,prunes,failed.Count);
    }
}

public static class ChoicePlanning
{
    public static int[][] Subsets(int n)
    {
        if(n<0||n>16)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();
        void Visit(int index)
        {
            if(index==n){result.Add(chosen.ToArray());return;}
            chosen.Add(index);Visit(index+1);chosen.RemoveAt(chosen.Count-1);
            Visit(index+1);
        }
        Visit(0);return result.ToArray();
    }
    public static int[][] Permutations(int n)
    {
        if(n<0||n>8)throw new ArgumentOutOfRangeException(nameof(n));
        var result=new List<int[]>();var chosen=new List<int>();var used=new bool[n];
        void Visit()
        {
            if(chosen.Count==n){result.Add(chosen.ToArray());return;}
            for(int i=0;i<n;i++)
            {
                if(used[i])continue;
                used[i]=true;chosen.Add(i);Visit();chosen.RemoveAt(chosen.Count-1);used[i]=false;
            }
        }
        Visit();return result.ToArray();
    }
}


raw calls=8191, bounds calls=3431, memo calls=85, hits=30
one-use state=(position, remaining); exhaustive feasible=False


## Project Connection

```{index} Project Connection
```

Specify a planning feature’s objective, inventory policy, state, transition, base cases, legal input range, and required witness. Explain whether sparse memoization, dense tabulation, or bounded search fits those constraints.

## Submit

```{index} Submit
```

Submit code, verified output, and explanations through the assignment panel when available.